<a href="https://colab.research.google.com/github/Angiehere/InteligenciaArtificial_y_RedesNeuronales_UANL_FIME/blob/main/ACTIVIDADES/PIA/Entrenamiento.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**PIA-Sistema mecatronico inteligente**



*  Angela Valeria Valdez Estrada
*  Yesenia Damaris Avila Cardona
*   Lesli Valeria Ibarra Beltran
*   Melissa Noemi Sanchez Ramos

Entrenamiento

In [ ]:

#lIBRERIAS

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import VotingClassifier
from sklearn.preprocessing import StandardScaler
import pickle

In [ ]:
ef aumentar(X, y, factor=10):
    X_aug, y_aug = [X], [y]
    for _ in range(factor):
        ruido = np.random.normal(0, 0.01, X.shape)
        X_aug.append(X + ruido)
        y_aug.append(y)
    return np.vstack(X_aug), np.concatenate(y_aug)

df = pd.read_csv("dataset.csv", header=None)
X = df.iloc[:, :-1].values
y = df.iloc[:, -1].values

print(f"Muestras originales: {len(X)}")
X, y = aumentar(X, y, factor=10)
print(f"Muestras tras aumento: {len(X)}")

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

mlp = MLPClassifier(hidden_layer_sizes=(32, 16), max_iter=1000, alpha=0.01, random_state=42)
arbol = DecisionTreeClassifier(max_depth=5, min_samples_leaf=5, random_state=42)
regresion = LogisticRegression(max_iter=1000, C=0.1, random_state=42)

ensemble = VotingClassifier(
    estimators=[('mlp', mlp), ('arbol', arbol), ('regresion', regresion)],
    voting='hard'
)

ensemble.fit(X_train, y_train)

print(f"\nPrecisión ensemble: {ensemble.score(X_test, y_test) * 100:.1f}%")
print("\nPrecisión por modelo:")
for nombre, modelo in ensemble.named_estimators_.items():
    print(f"  {nombre}: {modelo.score(X_test, y_test) * 100:.1f}%")

print("\nReglas del árbol de decisiones:")
print(export_text(ensemble.named_estimators_['arbol'], max_depth=3))

with open("modelo_mano.pkl", "wb") as f:
    pickle.dump(ensemble, f)
with open("scaler_mano.pkl", "wb") as f:
    pickle.dump(scaler, f)

print("Modelo guardado.")